# LpWM: dense state, sparse generator

This notebook runs the slot-free PushT or Wall experiment from `feature/sparse-generator`. It downloads the exact, unmodified DINO-WM datasets used by LpWM to Colab's local disk, while checkpoints, Hydra outputs, and W&B files are stored in MyDrive. The representation is a dense signed 8x8 patch field; exact top-k sparsity is used only for shared dynamics laws and token relations. Start with the smoke test before spending compute units.

In [ ]:
import os

if not os.path.exists('/content/lpworldmodel'):
    !git clone --branch feature/sparse-generator https://github.com/twojtys137/lpworldmodel.git /content/lpworldmodel
%cd /content/lpworldmodel
!git fetch origin feature/sparse-generator
!git checkout feature/sparse-generator
!git pull --ff-only origin feature/sparse-generator

In [ ]:
!pip -q install 'accelerate>=0.26,<2' 'hydra-core>=1.3,<2' 'omegaconf>=2.3,<3' 'wandb>=0.13,<1' 'submitit>=1.5,<2' 'hydra-submitit-launcher>=1.2,<2' einops decord pymunk pygame shapely scikit-image moviepy tensorboardX requests tqdm

# Planning imports are checked here, before any GPU work starts.
import submitit
import hydra_plugins.hydra_submitit_launcher
print(f'Planning dependencies: submitit {submitit.__version__}; Hydra launcher available')

!python -m pytest -q tests

## Exact LpWM data, MyDrive outputs, and W&B

Choose `pusht` (default) or `wall`. The downloader streams the corresponding original OSF archive to `/content/lpwm-data`, resumes interrupted downloads, verifies the official byte size and SHA-256, and extracts the original layout without conversion. PushT is 2.59 GiB compressed and Wall is 1.55 GiB compressed; the verified archive is removed after extraction. Local Colab storage is deliberately used for training speed.

In Colab, open **Secrets** (key icon), add a secret named `WANDB_API_KEY`, and grant this notebook access. Do not paste the key into a cell. The project `twojtys137-tw/lpwm-sparse-generator` is created automatically by the first authenticated run. By default, a missing or inaccessible secret stops setup before training; set `ALLOW_WANDB_OFFLINE = True` only when offline logging is intentional.

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata
import wandb

drive.mount('/content/drive')

ENV_NAME = 'pusht'  # change to 'wall' for the original Wall dataset
ALLOW_WANDB_OFFLINE = False  # avoid spending compute without cloud logging
DATASET_NAME = {'pusht': 'pusht_noise', 'wall': 'wall_single'}[ENV_NAME]
DATASET_DIR = Path('/content/lpwm-data')
RESULTS_DIR = Path('/content/drive/MyDrive/lpwm-sparse-generator')
WANDB_DIR = RESULTS_DIR  # W&B creates its own wandb/ subdirectory
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/download_lpwmdatasets.py',
    '--dataset', DATASET_NAME, '--output-dir', str(DATASET_DIR),
], check=True)

os.environ.update({
    'DATASET_DIR': str(DATASET_DIR),
    'ENV_NAME': ENV_NAME,
    'CKPT_BASE': str(RESULTS_DIR),
    'WANDB_DIR': str(WANDB_DIR),
    'WANDB_ENTITY': 'twojtys137-tw',
    'WANDB_PROJECT': 'lpwm-sparse-generator',
})

try:
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_MODE'] = 'online'
    login_ok = wandb.login(key=wandb_key, relogin=True)
    if login_ok is False:
        raise RuntimeError('W&B rejected WANDB_API_KEY.')
    print('W&B online: https://wandb.ai/twojtys137-tw/lpwm-sparse-generator')
else:
    os.environ['WANDB_MODE'] = 'offline'
    if not ALLOW_WANDB_OFFLINE:
        raise RuntimeError(
            'WANDB_API_KEY is missing or this notebook has no access to it. '
            'Add/grant the Colab secret and rerun this cell.'
        )
    print('W&B offline was explicitly allowed; sync the run before comparison.')
os.environ['REQUIRE_WANDB_ONLINE'] = '0' if ALLOW_WANDB_OFFLINE else '1'

print(f'Dataset: {DATASET_DIR / DATASET_NAME}')
print(f'Persistent results: {RESULTS_DIR}')

## 1. Smoke run

Eight rollouts, one epoch, batch 4 and 64 RDMReg projections. This validates data loading, gradients, checkpointing, and GPU memory.

In [ ]:
!SMOKE=1 bash scripts/train_sparse_generator_colab.sh

### Sync the newest offline run, if one exists

This is safe to run after adding the W&B secret. It finds the newest persisted `offline-run-*` directory (including runs created by an older version of this notebook) and uploads it to the configured entity/project. If the smoke run was already online, there is nothing to sync.

In [ ]:
offline_runs = sorted(
    (path for path in RESULTS_DIR.rglob('offline-run-*') if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
)
if not offline_runs:
    print('No persisted offline W&B run found; nothing to sync.')
elif os.environ.get('WANDB_MODE') != 'online':
    raise RuntimeError('Add/grant WANDB_API_KEY and rerun the setup cell first.')
else:
    latest_offline_run = offline_runs[-1]
    print(f'Syncing: {latest_offline_run}')
    subprocess.run([
        'wandb', 'sync',
        '--entity', os.environ['WANDB_ENTITY'],
        '--project', os.environ['WANDB_PROJECT'],
        str(latest_offline_run),
    ], check=True)

## 2. Screening run

The default is 50 rollouts, two epochs, batch 16, 64 patches, D=192, M=8, law top-k=2 and edge top-k=8. Set the flag only after the smoke run succeeds.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !SEED=0 bash scripts/train_sparse_generator_colab.sh

## 3. Controlled ablations

Keep the encoder and dense identity-linked state fixed. Compare `ltv`, `sparse_ltv`, `dense_generator`, and `sparse_generator`; promote only the best two to seeds 1 and 2. The detailed experiment matrix and interpretation of routing diagnostics are in `docs/sparse_generator.md`.

In [ ]:
# Print the matched commands; add RUN=1 (and initially SMOKE=1) to execute:
!bash scripts/sweep_sparse_generator_colab.sh
# !RUN=1 SMOKE=1 bash scripts/sweep_sparse_generator_colab.sh

# Example minimal sparse-LTV control using the same matched Colab config:
# !PREDICTOR=sparse_ltv NUM_PROJECTIONS=256 N_ROLLOUT=50 \
#   RUN_NAME=sparse_ltv_seed0 bash scripts/train_sparse_generator_colab.sh

## 4. Fair comparison with LpWM

This workflow separates the controlled 2x2 test (dense/sparse patch state × dense/sparse laws) from literal CLS+D384 LpWM/LeWM controls. It persists training metrics, planning logs, manifests and aggregate tables to `RESULTS_DIR`. Commands are dry-runs unless `RUN=1`; start with seed 0 and only then promote the matrix to three seeds.

In [ ]:
# Preview the four controlled screening cells (no training):
!PROFILE=screen STAGE=train SEEDS=0 bash scripts/benchmark_lpwm_sparse_generator_colab.sh

RUN_FAIR_SCREEN = False
if RUN_FAIR_SCREEN:
    !RUN=1 PROFILE=screen STAGE=train SEEDS=0 bash scripts/benchmark_lpwm_sparse_generator_colab.sh

# After seed 0 is checked, promote and plan on paired goals:
# !RUN=1 PROFILE=screen STAGE=train SEEDS="0 1 2" bash scripts/benchmark_lpwm_sparse_generator_colab.sh
# !RUN=1 PROFILE=screen STAGE=plan  SEEDS="0 1 2" bash scripts/benchmark_lpwm_sparse_generator_colab.sh
# !PROFILE=screen STAGE=collect bash scripts/benchmark_lpwm_sparse_generator_colab.sh

### Full-data confirmatory benchmark

The full profile uses the complete original PushT dataset and 8192 RDMReg projections. The `lpwm` and `lewm` cells use the paper's CLS+D384 Deep-AdaLN recipe; patch cells retain their matched D192 architecture. Raw latent errors are interpreted only within matched blocks, while PushT planning success is the cross-architecture endpoint. Keep the first command as a dry-run before enabling the expensive cells.

In [ ]:
FAIR_MODELS = 'lpwm lewm dense_dense dense_sparse sparse_dense sparse_sparse ltv sparse_ltv'
!PROFILE=full STAGE=train MODELS="$FAIR_MODELS" SEEDS="0 1 2" bash scripts/benchmark_lpwm_sparse_generator_colab.sh

# Uncomment sequentially after reviewing the dry-run:
# !RUN=1 PROFILE=full STAGE=train MODELS="$FAIR_MODELS" SEEDS="0 1 2" bash scripts/benchmark_lpwm_sparse_generator_colab.sh
# !RUN=1 PROFILE=full STAGE=plan  MODELS="$FAIR_MODELS" SEEDS="0 1 2" bash scripts/benchmark_lpwm_sparse_generator_colab.sh
# !PROFILE=full STAGE=collect bash scripts/benchmark_lpwm_sparse_generator_colab.sh

print(f'Results will be written under {RESULTS_DIR / "benchmarks" / "fair_lpwm_v1"}')